# 🎓 EduStream — Pipeline Medallion en Databricks

## 🎯 El escenario
**EduStream** es una plataforma de cursos online en **LATAM**. Cada día genera archivos
*batch* con datos de sus operaciones. El objetivo de este proyecto es diseñar el
**pipeline medallion completo** (Bronze → Silver → Gold) en Databricks: ingesta,
limpieza, validación de calidad, enriquecimiento y cálculo de métricas de negocio.

---

## 📥 Fuentes de datos (archivos batch diarios)

| Archivo CSV | Campos principales | Descripción y problemas de calidad |
|---|---|---|
| `enrollments.csv` | `enrollment_id`, `user_id`, `course_id`, `enrolled_at`, `payment_amount`, `currency` | Inscripciones de usuarios. ⚠️ Puede haber nulos en `payment_amount` (cursos gratuitos). |
| `courses.csv` | `course_id`, `title`, `instructor_id`, `category`, `price_usd`, `language` | Catálogo de cursos. ⚠️ Algunos tienen `category` en blanco. |
| `progress.csv` | `user_id`, `course_id`, `lessons_completed`, `total_lessons`, `last_activity_at` | Progreso por usuario y curso. ⚠️ Si `total_lessons = 0` es dato corrupto. |
| `instructors.csv` | `instructor_id`, `name`, `country`, `joined_at` | Datos de instructores. ⚠️ Algunos tienen `country = NULL`. |

---

## 🧹 Estrategia de limpieza por capa

| Capa | Responsabilidad |
|---|---|
| **Bronze** | Ingesta cruda de los CSV tal cual llegan (preserva el dato original). |
| **Silver** | Limpieza: drop de nulos críticos, reemplazo de nulos no críticos (`category → 'other'`, `country → 'unknown'`, `payment_amount → 0`), filtro de corruptos (`total_lessons > 0`), conversión de divisas a USD vía API. |
| **Gold** | Agregaciones y métricas de negocio listas para consumo. |

---

## 📊 Métricas de negocio requeridas (capa Gold)

| Métrica | Tabla Gold | Descripción |
|---|---|---|
| **Completion rate por curso** | `gold_completion_rate_by_course` | % de usuarios que completaron el 100% de las lecciones. |
| **Revenue por instructor** | `gold_revenue_by_instructor` | Suma de ingresos (USD) generados por los cursos de cada instructor. |
| **Cursos más abandonados** | `gold_top_abandoned_courses` | Cursos con alta inscripción pero bajo completion rate. |
| **Top categorías por revenue** | `gold_top_revenue_by_category` | Categorías que más ingresos generan por mes. |

---

## ⚡ Optimización de tablas (post-Gold)

Una vez creadas las tablas Gold, se aplican técnicas de mantenimiento de Delta Lake
para mejorar el rendimiento de lectura y controlar el almacenamiento:

| Comando | Qué hace | Aplicado en |
|---|---|---|
| **`OPTIMIZE`** | Compacta archivos pequeños en pocos archivos grandes | Gold |
| **`ZORDER`** | Co-localiza físicamente los datos por columnas de filtro frecuente (data skipping) | `enrolled_at`, `category`, `country` según la tabla |
| **`PARTITION BY`** | Divide físicamente los datos en carpetas por columna | `year` en `gold_top_revenue_by_category` |
| **`VACUUM`** | Elimina archivos de versiones antiguas para liberar almacenamiento | Todas las tablas |

---

## ⚡ Arquitectura Medallion
![Arquitectura Medallion](docs/EduStream-Arquitectura-Medallion.png)

---

## 🛠️ Stack tecnológico
- **Databricks Free Edition** + Unity Catalog
- **Delta Lake** — Time Travel, `OPTIMIZE`, `ZORDER`, `VACUUM`
- **Delta Live Tables (DLT)** — pipeline declarativo con expectativas de calidad
- **PySpark**
- **API** [open.er-api.com](https://open.er-api.com) — conversión de divisas locales a USD

---

## 🗂️ Contenido del proyecto
1. **Paso 1** — Lectura de CSVs y creación de tablas Delta (Bronze/Silver/Gold).
2. **Paso 2** — Time Travel: `DESCRIBE HISTORY`, `UPDATE`, consultas con `VERSION AS OF`.
3. **Paso 3** — Pipeline DLT con expectativas de calidad (notebook separado `edustream_pipeline_dlt`).
4. **Paso 4** — Optimización con `OPTIMIZE`, `ZORDER` y `VACUUM` justificados.

##Preparación del WORKSPACE

In [0]:
# ── Configuración inicial del catálogo ──────────────────────────
# Se usa IF NOT EXISTS para que la celda sea idempotente (re-ejecutable)
spark.sql('CREATE SCHEMA IF NOT EXISTS workspace.edustream')
display(spark.sql("SHOW SCHEMAS IN workspace"))

In [0]:
# ── Volumen de landing: zona de entrada de archivos CSV crudos ──
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.edustream.landing")
display(spark.sql("SHOW VOLUMES IN workspace.edustream"))

In [0]:
display(dbutils.fs.ls('/Volumes/workspace/edustream/landing'))

###Luego de crear el volumen, cargamos los archivos siguientes:
- enrollments.csv  —  inscripciones de usuarios a cursos
- courses.csv  —  catalogo de cursos
- progress.csv  —  progreso de cada usuario en cada curso
- instructors.csv  —  datos de los instructores

Los podemos generar ejecutando el archivo generate_data_edustream.py

# PASO 1: CREAR TABLAS DELTA EN UNITY CATALOG

In [0]:
from pyspark.sql.functions import (
    col, count, when, isnull,
    sum             as spark_sum,
    round           as spark_round,
    avg             as spark_avg,
    countDistinct   as count_distinct,
    to_timestamp,
    year, month, date_format
)
import requests
import json

In [0]:
# ── Reset del schema: elimina todas las tablas para re-ejecución limpia ──
# Permite que el notebook sea idempotente: si lo corres de nuevo, parte de cero
# sin tablas residuales de ejecuciones anteriores que puedan causar conflictos.
tables = spark.catalog.listTables('workspace.edustream')
display(spark.sql('SHOW TABLES IN workspace.edustream'))

for table in tables:
    full_name = f"workspace.edustream.{table.name}"
    
    if table.tableType == "VIEW":
        spark.sql(f"DROP VIEW IF EXISTS {full_name}")
        print(f"✅ VIEW eliminada:  {full_name}")
    else:
        spark.sql(f"DROP TABLE IF EXISTS {full_name}")
        print(f"✅ TABLE eliminada: {full_name}")

# Verificamos que quedó vacío
display(spark.sql('SHOW TABLES IN workspace.edustream'))

## ---------------------------------  BRONZE  ---------------------------------

In [0]:
# ── Bronze: lectura de CSVs y escritura como tablas Delta ────────
# Recorre cada CSV de landing y crea una tabla bronze_<nombre> automáticamente.
# inferSchema=True deja que Spark detecte los tipos de cada columna.
# mode('overwrite') reemplaza la tabla si ya existía (re-ejecutable).

files = [f.path for f in dbutils.fs.ls('/Volumes/workspace/edustream/landing') if f.path.endswith('.csv')]

for file_path in files:
    table_name = 'workspace.edustream.bronze_' + file_path.split('/')[-1].replace('.csv', '')
    df = spark.read.csv(file_path, header=True, inferSchema=True)
    df.write.format('delta').mode('overwrite').saveAsTable(table_name)
    print(f"✅ Bronze creada: {table_name}  →  {df.count()} filas")

display(spark.sql('SHOW TABLES IN workspace.edustream'))

## ---------------------------------  SILVER  ---------------------------------

In [0]:
# ── Silver Enrollments: limpieza de inscripciones ───────────────

df_silver_enrollments = spark.table('workspace.edustream.bronze_enrollments')

# Auditoría de nulos por columna
print("── Nulos por columna ──")
df_silver_enrollments.select([count(when(isnull(c), c)).alias(c) for c in df_silver_enrollments.columns]).show()
df_silver_enrollments = df_silver_enrollments.dropna(subset=['enrollment_id', 'user_id', 'course_id', 'enrolled_at'])

#reemplaza los nulos de payment_amount a 0 y currency a USD
df_silver_enrollments = df_silver_enrollments.fillna({'currency': 'USD', 'payment_amount': 0})

# Cast de fecha a tipo correcto
df_silver_enrollments = df_silver_enrollments.withColumn("enrolled_at", to_timestamp(col("enrolled_at")))

#muestra las estadisticas del dataset
df_silver_enrollments.describe().show()

#crear tabla delta
df_silver_enrollments.write.format('delta').mode("overwrite").saveAsTable("workspace.edustream.silver_enrollments")




In [0]:
# ── Silver Currency: tabla de tipos de cambio a USD ─────────────
# Fuente: open.er-api.com 

def get_exchange_rate(currency: str) -> float:
    if currency == 'USD':
        return 1.0
    try:
        url = f"https://open.er-api.com/v6/latest/{currency}"
        response = requests.get(url, timeout=5)
        return response.json()['rates']['USD']
    except (requests.RequestException, KeyError) as e:
        print(f"⚠️ Error con {currency}: {e}")
        return 1.0

df_silver_enrollments = spark.table('workspace.edustream.silver_enrollments')
currencies = [row.currency for row in df_silver_enrollments.select('currency').distinct().collect()]
rates_data = [(currency, get_exchange_rate(currency)) for currency in currencies]

df_silver_currency = spark.createDataFrame(rates_data, ['currency', 'exchange_rate'])
df_silver_currency.write.format('delta').mode("overwrite").saveAsTable("workspace.edustream.silver_currency")

display(df_silver_currency)

In [0]:
# ── Silver Progress: limpieza de progreso de usuarios ───────────

df_silver_progress = spark.table('workspace.edustream.bronze_progress')

#verificar si hay nulos en cada una de las columnas
print("── Nulos por columna ──")
df_silver_progress.select([count(when(isnull(c), c)).alias(c) for c in df_silver_progress.columns]).show()

#borro los registros que tienen total lessons igual a 0
df_silver_progress = df_silver_progress.filter(col("total_lessons") > 0)

#dropea los nulos de user_id, course_id, last_activity_at
df_silver_progress = df_silver_progress.dropna(subset=['user_id', 'course_id', 'last_activity_at'])

#reemplaza los nulos de lessons_completed a 0
df_silver_progress = df_silver_progress.fillna({'lessons_completed': 0})

# Cast de fecha
df_silver_progress = df_silver_progress.withColumn("last_activity_at", to_timestamp(col("last_activity_at")))

#muestra las estadisticas del dataset
df_silver_progress.describe().show()

#crear tabla delta
df_silver_progress.write.format('delta').mode("overwrite").saveAsTable("workspace.edustream.silver_progress")

print(f"✅ silver_progress: {df_silver_progress.count()} filas")


In [0]:
# ── Silver Courses: limpieza del catálogo de cursos ─────────────
df_silver_courses = spark.table('workspace.edustream.bronze_courses')

#verificar si hay nulos en cada una de las columnas
print("── Nulos por columna ──")
df_silver_courses.select([count(when(isnull(c), c)).alias(c) for c in df_silver_courses.columns]).show()

#reemplazar nulos de categoria por otros
df_silver_courses = df_silver_courses.fillna({'category': 'other'})

#muestra las estadisticas del dataset
df_silver_courses.describe().show()

#crear tabla delta
df_silver_courses.write.format('delta').mode("overwrite").saveAsTable("workspace.edustream.silver_courses")

print(f"✅ silver_courses: {df_silver_courses.count()} filas")

In [0]:
# ── Silver Instructors: limpieza de datos de instructores ───────
df_silver_instructors = spark.table('workspace.edustream.bronze_instructors')

#verificar si hay nulos en cada una de las columnas
print("── Nulos por columna ──")
df_silver_instructors.select([count(when(isnull(c), c)).alias(c) for c in df_silver_instructors.columns]).show()

#reemplazar nulos de country por otros
df_silver_instructors = df_silver_instructors.fillna({'country': 'Unknown'})

# Cast de fecha de ingreso
df_silver_instructors = df_silver_instructors.withColumn("joined_at", to_timestamp(col("joined_at")))

#muestra las estadisticas del dataset
df_silver_instructors.describe().show()

#crear tabla delta
df_silver_instructors.write.format('delta').mode("overwrite").saveAsTable("workspace.edustream.silver_instructors")

print(f"✅ silver_instructors: {df_silver_instructors.count()} filas")

In [0]:
# ── Silver Enrollments Clean: conversión a USD y enriquecimiento ─
df_silver_enrollments = spark.table('workspace.edustream.silver_enrollments')
df_silver_currency = spark.table('workspace.edustream.silver_currency')
df_silver_courses = spark.table('workspace.edustream.silver_courses')
df_silver_instructors = spark.table('workspace.edustream.silver_instructors')

# 1. Convertir payment_amount a USD usando tabla de tasas
df_silver_enrollments_clean = (
    df_silver_enrollments
    .join(df_silver_currency, on =['currency'], how = 'left')
    .withColumn('payment_USD', spark_round(col('payment_amount') * col('exchange_rate'), 6))
    .drop('payment_amount', 'exchange_rate')
)

# 2. Enriquecer con datos del curso e instructor

df_silver_enrollments_clean = df_silver_enrollments_clean.join(df_silver_courses, on = ['course_id'], how = 'left')
df_silver_enrollments_clean = df_silver_enrollments_clean.join(df_silver_instructors, on = ['instructor_id'], how = 'left')

#muestra las estadisticas del dataset
df_silver_enrollments_clean.describe().show()
display(df_silver_enrollments_clean)

#crear tabla delta
df_silver_enrollments_clean.write.format('delta').mode('overwrite').saveAsTable('workspace.edustream.silver_enrollments_clean')
print(f"✅ silver_enrollments_clean: {df_silver_enrollments_clean.count()} filas")

In [0]:
# ── Silver Progress Clean: cálculo de completion_rate ───────────
df_silver_progress = spark.table('workspace.edustream.silver_progress')
df_silver_courses = spark.table('workspace.edustream.silver_courses')

# completion_rate = lecciones completadas / total lecciones
df_silver_progress_clean = df_silver_progress.withColumn('completion_rate', spark_round(col('lessons_completed') / col('total_lessons'), 4))

# Enriquecer con información del curso
df_silver_progress_clean = df_silver_progress_clean.join(df_silver_courses, on=["course_id"], how="left")

#muestra las estadisticas del dataset
df_silver_progress_clean.describe().show()
display(df_silver_progress_clean)

#crear tabla delta
df_silver_progress_clean.write.format('delta').mode('overwrite').saveAsTable('workspace.edustream.silver_progress_clean')
print(f"✅ silver_progress_clean: {df_silver_progress_clean.count()} filas")




## ---------------------------------  GOLD  ---------------------------------

In [0]:
# ── Gold: tasa de finalización por curso ────────────────────────
# Métrica: % de usuarios que completaron el 100% de las lecciones

df_silver_progress_clean = spark.table('workspace.edustream.silver_progress_clean')

df_gold_completion_rate_by_course = df_silver_progress_clean \
    .groupBy('course_id', 'title', 'category') \
    .agg(
        count('user_id').alias('total_users'),
        spark_sum(when(col('completion_rate') == 1.0, 1)
            .otherwise(0)).alias('total_users_completed')
    ) \
    .withColumn(
        'pct_users_completed',
        spark_round(col('total_users_completed') / col('total_users') * 100, 2)
    ) \
    .orderBy(col('pct_users_completed').desc())

display(df_gold_completion_rate_by_course)

df_gold_completion_rate_by_course.write.format('delta').mode('overwrite').saveAsTable('workspace.edustream.gold_completion_rate_by_course')

In [0]:
# ── Gold: revenue total generado por instructor ──────────────────
# Métrica: suma de ingresos en USD de todos los cursos de cada instructor

df_silver_enrollments_clean = spark.table("workspace.edustream.silver_enrollments_clean")

df_gold_revenue_by_instructor = (
    df_silver_enrollments_clean
    .groupBy("instructor_id", "name", "country")
    .agg(
        spark_sum("payment_USD").alias("total_revenue_USD"),
        count("enrollment_id").alias("total_enrollments"),
        count_distinct("course_id").alias("total_courses")
    )
    .withColumn("total_revenue_USD", spark_round(col("total_revenue_USD"), 6))
    .orderBy(col("total_revenue_USD").desc())
)

display(df_gold_revenue_by_instructor)

df_gold_revenue_by_instructor.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.edustream.gold_revenue_by_instructor")

print(f"✅ gold_revenue_by_instructor: {df_gold_revenue_by_instructor.count()} filas")

In [0]:
# ── Gold: cursos con alta inscripción y bajo completion rate ─────
# Se requiere mínimo de inscripciones para que la métrica sea estadísticamente relevante

df_silver_progress_clean    = spark.table("workspace.edustream.silver_progress_clean")
df_silver_enrollments_clean = spark.table("workspace.edustream.silver_enrollments_clean")

# 1. Métricas de progreso por curso
df_progress_by_course = (
    df_silver_progress_clean
    .groupBy("course_id", "title", "category")
    .agg(
        spark_avg(col("completion_rate")).alias("avg_completion_rate"),
        count("user_id").alias("users_with_progress")
    )
)

# 2. Total de inscripciones por curso
df_enrollments_by_course = (
    df_silver_enrollments_clean
    .groupBy("course_id")
    .agg(count("enrollment_id").alias("total_enrolled"))
)

# 3. Join + cálculo de tasa de abandono
df_gold_top_abandoned_courses = (
    df_progress_by_course
    .join(df_enrollments_by_course, on=["course_id"], how="left")
    .withColumn("avg_completion_rate", spark_round(col("avg_completion_rate"), 4))
    .withColumn("abandonment_rate",    spark_round(1 - col("avg_completion_rate"), 4))
    .filter(col("total_enrolled") >= 10)   # umbral mínimo de relevancia estadística
    .orderBy(
        col("abandonment_rate").desc(),
        col("total_enrolled").desc()
    )
)

display(df_gold_top_abandoned_courses)

df_gold_top_abandoned_courses.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.edustream.gold_top_abandoned_courses")

print(f"✅ gold_top_abandoned_courses: {df_gold_top_abandoned_courses.count()} filas")

In [0]:
# ── Gold: top categorías por ingresos agrupados por mes ─────────
# Métrica: suma de payment_USD por categoría y mes de inscripción

df_silver_enrollments_clean = spark.table("workspace.edustream.silver_enrollments_clean")

df_gold_top_revenue_by_category = (
    df_silver_enrollments_clean
    .filter(col("enrolled_at").isNotNull())            # excluir fechas nulas
    .withColumn("year",        year(col("enrolled_at")))
    .withColumn("month",       month(col("enrolled_at")))
    .withColumn("month_label", date_format(col("enrolled_at"), "yyyy-MM"))
    .groupBy("year", "month", "month_label", "category")
    .agg(
        spark_sum("payment_USD").alias("total_revenue_USD"),
        count("enrollment_id").alias("total_enrollments"),
        spark_avg("payment_USD").alias("avg_revenue_per_enrollment")
    )
    .withColumn("total_revenue_USD",          spark_round(col("total_revenue_USD"), 2))
    .withColumn("avg_revenue_per_enrollment", spark_round(col("avg_revenue_per_enrollment"), 2))
    .orderBy(
        col("year").desc(),
        col("month").desc(),
        col("total_revenue_USD").desc()
    )
)

display(df_gold_top_revenue_by_category)

df_gold_top_revenue_by_category.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.edustream.gold_top_revenue_by_category")

print(f"✅ gold_top_revenue_by_category: {df_gold_top_revenue_by_category.count()} filas")

## REFLEXIÓN 1
1.  Cuando ejecutaste saveAsTable, Databricks creo una tabla 'managed' en Unity Catalog. Que diferencia hay entre una managed table y una external table? Cual elegirias para los datos de EduStream y por que?

    La managed table se crea dentro del catalogo Databricks y la externa es una tabla que se guarda dentro de un contenedor externo (S3, ADLS,GCS)
    la managed table se puede borrar la data y metadata, en la externa solo la metadata.
    Para guarda una tabla externa, se debe especificar el path.

## PASO 2 - EXPLORAR TIME TRAVEL CON TABLAS DELTA

In [0]:
%sql
-- Estado inicial: vemos los primeros registros ANTES de modificar nada.
-- Esto será la "versión 0" a la que volveremos con Time Travel.
SELECT * FROM workspace.edustream.silver_enrollments_clean ORDER BY enrollment_id LIMIT 5;

In [0]:
%sql
-- Modificación 1: cambiamos el país de 2 registros.
-- Cada UPDATE genera una NUEVA versión de la tabla Delta (queda registrada en el historial).
UPDATE workspace.edustream.silver_enrollments_clean
SET    country = 'España'
WHERE  enrollment_id IN ('ENR000001', 'ENR000002');

SELECT * FROM workspace.edustream.silver_enrollments_clean ORDER BY enrollment_id LIMIT 5;

In [0]:
%sql
-- contar las filas de silver enrollments clean
SELECT COUNT(*) FROM workspace.edustream.silver_enrollments_clean

In [0]:
%sql
-- Modificación 2: aumentamos todos los pagos en 10%.
-- Genera otra versión más, simulando un cambio masivo de datos.
UPDATE workspace.edustream.silver_enrollments_clean
SET    payment_USD = payment_USD* 1.1



In [0]:
%sql
-- DESCRIBE HISTORY lista TODAS las versiones de la tabla con su timestamp, operación
-- (WRITE, UPDATE...) y usuario. Es el registro de auditoría que habilita el Time Travel.
DESCRIBE HISTORY workspace.edustream.silver_enrollments_clean

In [0]:
%sql
-- Time Travel: consultamos la tabla tal como estaba en la versión 0 (estado original),
-- antes de los dos UPDATE. Útil para auditar o recuperar datos
SELECT * FROM workspace.edustream.silver_enrollments_clean
VERSION AS OF 0
LIMIT 5
    

In [0]:
%sql
-- Comparación lado a lado: une la versión ACTUAL con la versión 0 por enrollment_id
-- para ver la diferencia y el % de cambio en payment_USD tras el aumento del 10%.
-- (El CASE evita dividir entre cero cuando payment_USD_v0 = 0)
SELECT  actual.enrollment_id,
        v0.payment_USD                              AS payment_USD_v0,
        actual.payment_USD                          AS payment_USD_actual,
         ROUND(actual.payment_USD - v0.payment_USD , 4)        AS diferencia,
        CASE 
            WHEN v0.payment_USD > 0 THEN 
            ROUND((actual.payment_USD / v0.payment_USD - 1) * 100, 2)
        ELSE 0 
        END  AS pct_cambio

FROM    workspace.edustream.silver_enrollments_clean            AS actual
JOIN    workspace.edustream.silver_enrollments_clean
        VERSION AS OF 0                                         AS v0
        ON actual.enrollment_id = v0.enrollment_id
ORDER BY actual.enrollment_id
LIMIT 10;


## REFLEXIÓN 2

2.  En el mundo real, una tabla puede tener cientos de versiones acumuladas con el tiempo. Cual seria el problema de no limpiar versiones antiguas? Que comando de Delta Lake resuelve esto (lo veras en el Paso 4)?

    No limpiar las versiones antiguas puede ocasionar un sobre almacenamiento de versiones que no se van a utilizar, y a mayor almacenamiento utilizado, mayor costo asociado. 
    VACUUM nos ayuda a limpiar las versiones anteriores definiendo tiempos de reseteo

## PASO 3 - CREAR UN PIPELINE DLT CON EXPECTATIVAS DE CALIDAD

## Pipeline DLT

El pipeline declarativo está en el notebook separado **`edustream_pipeline_dlt`**.
Implementa la arquitectura medallion completa con **expectativas de calidad** (`@dlt.expect_all_or_drop`)
en cada tabla Silver.

### Grafo del pipeline en ejecución
![Grafo DLT](docs/pipeline_edustream_DAG.png)

## REFLEXIÓN 3

2.  En este paso usaste expect_or_drop para payment_amount negativo y expect normal para course_id NULL. Por que esa diferencia? En que caso usarias expect_or_fail en su lugar?

    expect nos ayuda a identificar y registrar una posible falla o problema
    expect_or_drop nos ayuda a identificar y borrar las posibles fallas
    expect_or_fail si no se cumplen las condiciones detiene el pipeline

    en este caso, para el payment_amount, se elige dropear ya que los valores negativos son errores que no necesitamos
    para el caso de course_id NULL, solo se registra el error, ya que podría tener información financiera que no se debe perder.
    en el caso de fail es cuando ese dato es tan importante que sería mejor parar, por ejemplo si el archivo de cursos está vacío

## PASO 4 - OPTIMIZAR TABLAS CON OPTIMIZE, ZORDER y VACUUM

Delta Lake permite optimizar el almacenamiento y la velocidad de lectura mediante 3 comandos:

- **`OPTIMIZE`** → compacta muchos archivos pequeños en pocos archivos grandes (mejora lectura).
- **`ZORDER`** → co-localiza físicamente los datos por columnas de filtro frecuente (data skipping).
- **`VACUUM`** → elimina archivos de versiones antiguas para liberar almacenamiento y reducir costos.

 💡 Estos comandos NO van dentro del pipeline DLT. Se ejecutan como mantenimiento posterior.

In [0]:
%sql
-- ───────────────────────────────────────────────────────────────────
-- DEMOSTRAR FUNCIONAMIENTO DE OPTIMIZE
-- ───────────────────────────────────────────────────────────────────
-- Si usamos la tabla original, OPTIMIZE no tendría nada que compactar.
-- Para DEMOSTRAR el efecto, duplicamos los datos 3 veces: cada INSERT genera un archivo Parquet nuevo.
-- Simulando el escenario real donde llegan archivos batch diarios que fragmentan la tabla.
-- Creamos una copia SOLO para demostrar el funcionamiento de OPTIMIZE (sin modificar la tabla real)

CREATE OR REPLACE TABLE workspace.edustream.demo_optimize
AS SELECT * FROM workspace.edustream.silver_enrollments_clean;

INSERT INTO workspace.edustream.demo_optimize SELECT * FROM workspace.edustream.demo_optimize;
INSERT INTO workspace.edustream.demo_optimize SELECT * FROM workspace.edustream.demo_optimize;
INSERT INTO workspace.edustream.demo_optimize SELECT * FROM workspace.edustream.demo_optimize;


In [0]:
%sql
-- ───────────────────────────────────────────────────────────────────
-- ESTADO ANTES de optimizar
-- ───────────────────────────────────────────────────────────────────
-- DESCRIBE DETAIL muestra metadatos físicos de la tabla Delta.
-- Fijarse en la columna `numFiles`: indica cuántos archivos Parquet componen
-- la tabla. Tras los INSERT debería mostrar varios archivos (ej. 4).

DESCRIBE DETAIL workspace.edustream.demo_optimize;

In [0]:
%sql
-- ───────────────────────────────────────────────────────────────────
-- OPTIMIZE + ZORDER
-- ───────────────────────────────────────────────────────────────────
-- OPTIMIZE compacta los archivos pequeños en uno grande (~1 GB ideal).
-- ZORDER BY (enrolled_at) reordena físicamente los datos por fecha de inscripción.
--
-- ¿Por qué enrolled_at? Porque las consultas analíticas sobre inscripciones casi
-- siempre filtran por rango de fechas (tendencias mensuales, inscripciones recientes).
-- Co-localizar por esta columna permite a Spark saltarse archivos que no contienen
-- el rango buscado (data skipping), reduciendo los datos leídos.

OPTIMIZE workspace.edustream.demo_optimize
ZORDER BY enrolled_at;

-- ESTADO DESPUÉS: numFiles debería bajar (ej. de 4 → 1), confirmando la compactación.
DESCRIBE DETAIL workspace.edustream.demo_optimize;

In [0]:
%sql
-- ───────────────────────────────────────────────────────────────────
-- VACUUM
-- ───────────────────────────────────────────────────────────────────
-- Cada operación (INSERT, UPDATE, OPTIMIZE) deja archivos antiguos en disco para
-- permitir Time Travel. VACUUM elimina los archivos más antiguos que el período
-- de retención (7 días por defecto) que ya no son referenciados por la versión actual.
-- Esto libera almacenamiento y reduce costos.
VACUUM workspace.edustream.demo_optimize;

-- DRY RUN lista qué archivos se borrarían SIN borrarlos (modo simulación / auditoría).
-- Útil para validar antes de ejecutar un VACUUM real en producción.
VACUUM workspace.edustream.demo_optimize DRY RUN;

## REFLEXIÓN 4

4.  Por que ZORDER por enrolled_at (o la columna que hayas elegido) tiene sentido para silver_enrollments? Si tuvieras que aplicar ZORDER a gold_completion_rate, que columna elegirias y por que?

    ZORDER nos ayuda a ordenar y segmentar las búsquedas que se vayan a realizar.
    Se aplica a silver enrollments para ordenar las fechas más recientes.
    Para el caso de gold_completion_rate se elegiría la columna category para ordenar la busqueda de cursos 

In [0]:
# Recrear gold_top_revenue_by_category con partición por year
df_gold_top_revenue_by_category = spark.table('workspace.edustream.gold_top_revenue_by_category')

df_gold_top_revenue_by_category.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .partitionBy('year') \
    .saveAsTable('workspace.edustream.gold_top_revenue_by_category')

print('✅ Tabla recreada con partición por year')

In [0]:
%sql
-- ───────────────────────────────────────────────────────────────────
-- OPTIMIZE de las tablas GOLD (capa de consumo)
-- ───────────────────────────────────────────────────────────────────
-- Las tablas Gold alimentan dashboards y reportes, donde la velocidad de lectura
-- es crítica. Se elige el ZORDER según la columna de filtro más frecuente de cada métrica.


-- gold_revenue_by_instructor
-- ZORDER por country: tabla pequeña (~151 filas, una por instructor).
-- Se ordena por country porque los análisis de revenue suelen segmentarse por país.
OPTIMIZE workspace.edustream.gold_revenue_by_instructor
ZORDER BY (country);

-- gold_top_revenue_by_category
-- ZORDER por (category, month): las búsquedas filtran ingresos por categoría
-- y período temporal, así que co-localizar por ambas columnas optimiza el acceso.
OPTIMIZE workspace.edustream.gold_top_revenue_by_category
ZORDER BY (category, month);

-- gold_top_abandoned_courses
-- ZORDER por category: se consultan los cursos más abandonados filtrando
-- por categoría para detectar qué áreas tienen peor retención.
OPTIMIZE workspace.edustream.gold_top_abandoned_courses
ZORDER BY (category);

-- gold_completion_rate_by_course
-- ZORDER por category: las consultas de dashboards filtran cursos por categoría,
-- co-localizar por esta columna acelera esas búsquedas.
OPTIMIZE workspace.edustream.gold_completion_rate_by_course
ZORDER BY (category);

In [0]:
%sql
-- VACUUM: elimina archivos de versiones antiguas (>7 días por defecto)
-- para liberar almacenamiento y reducir costos.
VACUUM workspace.edustream.gold_completion_rate_by_course;
VACUUM workspace.edustream.gold_revenue_by_instructor;
VACUUM workspace.edustream.gold_top_abandoned_courses;
VACUUM workspace.edustream.gold_top_revenue_by_category;